In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [3]:
class SelfAttention(nn.Module):
    def __init__(self, d_model = 2, row_dim = 0, col_dim = 1):
        
        super().__init__()
        
        self.W_q = nn.Linear(in_features = d_model, out_features = d_model, bias = False)
        self.W_k = nn.Linear(in_features = d_model, out_features = d_model, bias = False)
        self.W_v = nn.Linear(in_features = d_model, out_features = d_model, bias = False)
        
        self.row_dim = row_dim
        self.col_dim = col_dim
        
    def forward(self, token_encodings):
        
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)
        
        # Compute attention scores
        attention_scores = torch.matmul(q, k.transpose(dim0 = self.row_dim, dim1 = self.col_dim))
        
        # Apply softmax to get attention weights
        attention_weights = F.softmax(attention_scores, dim = self.col_dim) 
        
        # Compute the weighted sum of the values
        output = torch.matmul(attention_weights, v)
        
        return output
    
    
    
    

In [4]:
## create a matrix of token encodings...
encodings_matrix = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])

## set the seed for the random number generator
torch.manual_seed(42)

In [5]:
self_attention = SelfAttention(d_model = 2, row_dim = 0, col_dim = 1)

self_attention(token_encodings = encodings_matrix)

tensor([[0.8777, 1.0034],
        [0.0313, 0.6368],
        [3.7436, 2.3622]], grad_fn=<MmBackward0>)

In [7]:
self_attention.W_q.weight.transpose(0, 1)

tensor([[ 0.5406, -0.1657],
        [ 0.5869,  0.6496]], grad_fn=<TransposeBackward0>)

In [8]:
self_attention.W_k.weight.transpose(0, 1)

tensor([[-0.1549, -0.3443],
        [ 0.1427,  0.4153]], grad_fn=<TransposeBackward0>)

In [9]:
self_attention.W_v.weight.transpose(0, 1)

tensor([[ 0.6233,  0.6146],
        [-0.5188,  0.1323]], grad_fn=<TransposeBackward0>)

In [10]:
self_attention.W_q(encodings_matrix)

tensor([[ 0.7621, -0.0428],
        [ 1.1063,  0.7890],
        [ 1.1164, -2.1336]], grad_fn=<MmBackward0>)

## Masked Self Attention

In [11]:
class MaskedSelfAttention(nn.Module):
    
    def __init__(self, d_model = 2, row_dim = 0, col_dim = 1):
        
        super().__init__()
        
        self.W_q = nn.Linear(in_features = d_model, out_features = d_model, bias = False)
        self.W_k = nn.Linear(in_features = d_model, out_features = d_model, bias = False)
        self.W_v = nn.Linear(in_features = d_model, out_features = d_model, bias = False)
        
        self.row_dim = row_dim
        self.col_dim = col_dim
        
    def forward(self, token_encodings, mask = None):
        
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)
        
        attention_scores = torch.matmul(q, k.transpose(dim0 = self.row_dim, dim1 = self.col_dim))
        
        if mask is not None:
            ## Here we are masking out things we don't want to pay attention to
            ##
            ## We replace values we wanted masked out
            ## with a very small negative number so that the SoftMax() function
            ## will give all masked elements an output value (or "probability") of 0.
            attention_scores = attention_scores.masked_fill(mask=mask, value=-1e9) # I've also seen -1e20 and -9e15 used in masking
            
        attention_weights = F.softmax(attention_scores, dim = self.col_dim)
        
        output = torch.matmul(attention_weights, v)
        
        return output
    
        
            

In [13]:
encoding_matrix = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])


torch.manual_seed(42)


masked_attention = MaskedSelfAttention(d_model = 2, row_dim = 0, col_dim = 1)

mask = torch.tril(torch.ones(3, 3))

mask = mask == 0 
mask

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [14]:
masked_attention(encoding_matrix, mask)

tensor([[ 0.6038,  0.7434],
        [-0.0565,  0.5959],
        [ 3.7436,  2.3622]], grad_fn=<MmBackward0>)

In [38]:
import torch

# experiment settings
d = 5
nlayers = 100
normalize = False # set True to use normalization

# create vector with random entries between [-1, 1]
input_vector = (torch.rand(d) - 0.5) * 2.0
print(input_vector)

# create matrix with random entries between [-1, 1]
# by which we can repeatedly multiply the input vector
weight_matrix = (torch.rand(d, d) - 0.5) * 2.0

output = input_vector
for i in range(nlayers):
    # optionally perform normalization
    if normalize:
        output = (output - torch.mean(output)) / torch.std(output)

    # repeatedly multiply the vector by the matrix
    output = weight_matrix @ output

# observe output values
print(output)

tensor([-0.4772, -0.8387,  0.2511, -0.8105,  0.4224])
tensor([-4.1381e+17, -7.8146e+18,  5.7571e+18,  6.7882e+18, -1.3900e+19])
